# Validate Databases

In [1]:
import sqlite3
import pandas as pd

## Keys unique and not missing

In [2]:
# 1 Check keys are unique and not missing

keys = {
    "asset_race": ["season", "race_number", "asset_id"],
    "team_race": ["season", "race_number", "user_guid", "team_no"],
    "team_race_asset": [
        "season", "race_number", "user_guid", "team_no", "asset_id"
    ],
    "team_chip_usage": ["season", "user_guid", "team_no", "chip_id"],
    "races": ["season", "race_number"],
    "chips": ["chip_id"],
}

In [3]:
results = []

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    for table, key_columns in keys.items():
        columns = ", ".join(key_columns)
        df = pd.read_sql_query(f"SELECT {columns} FROM {table}", con)

        results.append({
            "table": table,
            "rows": len(df),
            "rows_with_missing_keys": df.isna().any(axis=1).sum(),
            "duplicate_key_rows": df.duplicated(keep=False).sum(),
        })

pd.DataFrame(results)

,table,rows,rows_with_missing_keys,duplicate_key_rows
0,asset_race,538,0,0
1,team_race,225,0,0
2,team_race_asset,1576,0,0
3,team_chip_usage,25,0,0
4,races,22,0,0
5,chips,6,0,0


## Table Relationships

In [4]:
# 2. check relationships

relationship_checks = {
    "team assets → team race": """
        SELECT COUNT(*)
        FROM team_race_asset a
        WHERE NOT EXISTS (
            SELECT 1 FROM team_race t
            WHERE t.season = a.season
              AND t.race_number = a.race_number
              AND t.user_guid = a.user_guid
              AND t.team_no = a.team_no
        )
    """,
    "team assets → asset race": """
        SELECT COUNT(*)
        FROM team_race_asset t
        WHERE NOT EXISTS (
            SELECT 1 FROM asset_race a
            WHERE a.season = t.season
              AND a.race_number = t.race_number
              AND a.asset_id = t.asset_id
        )
    """,
    "team race → races": """
        SELECT COUNT(*)
        FROM team_race t
        WHERE NOT EXISTS (
            SELECT 1 FROM races r
            WHERE r.season = t.season
              AND r.race_number = t.race_number
        )
    """,
    "asset race → races": """
        SELECT COUNT(*)
        FROM asset_race a
        WHERE NOT EXISTS (
            SELECT 1 FROM races r
            WHERE r.season = a.season
              AND r.race_number = a.race_number
        )
    """,
    "chip usage → chips": """
        SELECT COUNT(*)
        FROM team_chip_usage u
        WHERE NOT EXISTS (
            SELECT 1 FROM chips c
            WHERE c.chip_id = u.chip_id
        )
    """,
    "chip usage → team race used": """
        SELECT COUNT(*)
        FROM team_chip_usage u
        WHERE NOT EXISTS (
            SELECT 1 FROM team_race t
            WHERE t.season = u.season
              AND t.race_number = u.race_used
              AND t.user_guid = u.user_guid
              AND t.team_no = u.team_no
        )
    """,
}

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    results = [
        {
            "relationship": name,
            "unmatched_rows": con.execute(query).fetchone()[0],
        }
        for name, query in relationship_checks.items()
    ]

pd.DataFrame(results)

,relationship,unmatched_rows
0,team assets → team race,0
1,team assets → asset race,0
2,team race → races,0
3,asset race → races,0
4,chip usage → chips,0
5,chip usage → team race used,1


In [5]:
query = """
    SELECT u.*
    FROM team_chip_usage u
    WHERE NOT EXISTS (
        SELECT 1
        FROM team_race t
        WHERE t.season = u.season
          AND t.race_number = u.race_used
          AND t.user_guid = u.user_guid
          AND t.team_no = u.team_no
    )
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    unmatched_chip = pd.read_sql_query(query, con)

unmatched_chip

,season,user_guid,team_no,team_name,chip_id,race_used
0,2026,ddb624fa-1369-11f1-9c45-25f49ad9af08-0-231975470,1,Trulli madly deeply,4,16


In [6]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    coverage = pd.read_sql_query("""
        SELECT season, MIN(race_number) AS first_race,
               MAX(race_number) AS latest_race,
               COUNT(DISTINCT race_number) AS races_loaded
        FROM team_race
        GROUP BY season
    """, con)

coverage

,season,first_race,latest_race,races_loaded
0,2026,1,15,15


In [7]:
chip_checks = {
    "chip usage → known team": """
        SELECT COUNT(*)
        FROM team_chip_usage u
        WHERE NOT EXISTS (
            SELECT 1 FROM team_race t
            WHERE t.season = u.season
              AND t.user_guid = u.user_guid
              AND t.team_no = u.team_no
        )
    """,
    "chip usage → known race": """
        SELECT COUNT(*)
        FROM team_chip_usage u
        WHERE NOT EXISTS (
            SELECT 1 FROM races r
            WHERE r.season = u.season
              AND r.race_number = u.race_used
        )
    """,
    "historical chip usage → team race": """
        SELECT COUNT(*)
        FROM team_chip_usage u
        WHERE u.race_used <= (
            SELECT MAX(t.race_number)
            FROM team_race t
            WHERE t.season = u.season
        )
        AND NOT EXISTS (
            SELECT 1 FROM team_race t
            WHERE t.season = u.season
              AND t.race_number = u.race_used
              AND t.user_guid = u.user_guid
              AND t.team_no = u.team_no
        )
    """,
}

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    results = [
        {
            "relationship": name,
            "unmatched_rows": con.execute(query).fetchone()[0],
        }
        for name, query in chip_checks.items()
    ]

pd.DataFrame(results)

,relationship,unmatched_rows
0,chip usage → known team,0
1,chip usage → known race,0
2,historical chip usage → team race,0


## Teams with unusual amount of assets in a given race

In [8]:
query = """
    SELECT
        season,
        race_number,
        user_guid,
        team_no,
        team_name,
        COUNT(*) AS asset_count
    FROM team_race_asset
    GROUP BY season, race_number, user_guid, team_no, team_name
    HAVING COUNT(*) != 7
    ORDER BY race_number, team_no
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    unusual_rosters = pd.read_sql_query(query, con)

unusual_rosters

,season,race_number,user_guid,team_no,team_name,asset_count
0,2026,15,f4f4335a-13f5-11f1-ab22-83c15a3ae2b2-0-175781919,1,Gorąco Polecam,8


In [9]:
query = """
    SELECT
        t.asset_id,
        a.display_name,
        a.asset_type,
        t.is_final,
        t.player_position,
        t.is_captain,
        t.is_mg_captain
    FROM team_race_asset t
    JOIN asset_race a
      ON a.season = t.season
     AND a.race_number = t.race_number
     AND a.asset_id = t.asset_id
    WHERE t.season = 2026
      AND t.race_number = 15
      AND t.user_guid = 'f4f4335a-13f5-11f1-ab22-83c15a3ae2b2-0-175781919'
      AND t.team_no = 1
    ORDER BY t.player_position, t.is_final
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    final_fix_assets = pd.read_sql_query(query, con)

final_fix_assets

,asset_id,display_name,asset_type,is_final,player_position,is_captain,is_mg_captain
0,11161,K. Antonelli,DRIVER,0,1,0,0
1,111,N. Hulkenberg,DRIVER,0,2,0,0
2,12,F. Alonso,DRIVER,0,3,0,0
3,129,L. Stroll,DRIVER,0,4,0,0
4,11051,G. Bortoleto,DRIVER,2,5,0,0
5,25,Ferrari,CONSTRUCTOR,0,6,0,0
6,27,McLaren,CONSTRUCTOR,0,7,0,0
7,18,P. Gasly,DRIVER,1,8,1,0


In [10]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    final_status_counts = pd.read_sql_query("""
        SELECT is_final, COUNT(*) AS rows
        FROM team_race_asset
        GROUP BY is_final
        ORDER BY is_final
    """, con)

final_status_counts

,is_final,rows
0,0,1574
1,1,1
2,2,1


## Missing Values in Analytical Fields

In [11]:
columns_to_check = {
    "asset_race": [
        "asset_type", "value", "gameday_points"
    ],
    "team_race": [
        "race_points", "race_rank",
        "recorded_asset_value", "remaining_budget",
        "recorded_total_budget", "transfers_made",
        "free_transfers_remaining"
    ],
    "team_race_asset": [
        "is_captain", "is_mg_captain",
        "player_position", "is_final"
    ],
    "team_chip_usage": ["race_used"],
}

In [12]:
results = []

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    for table, columns in columns_to_check.items():
        df = pd.read_sql_query(
            f"SELECT {', '.join(columns)} FROM {table}", con
        )

        for column in columns:
            results.append({
                "table": table,
                "column": column,
                "missing_rows": df[column].isna().sum(),
                "total_rows": len(df),
            })

pd.DataFrame(results)

,table,column,missing_rows,total_rows
0,asset_race,asset_type,0,538
1,asset_race,value,0,538
2,asset_race,gameday_points,0,538
3,team_race,race_points,0,225
4,team_race,race_rank,0,225
5,team_race,recorded_asset_value,97,225
6,team_race,remaining_budget,0,225
7,team_race,recorded_total_budget,97,225
8,team_race,transfers_made,0,225
9,team_race,free_transfers_remaining,0,225


In [13]:
query = """
    SELECT
        race_number,
        COUNT(*) AS team_count,
        SUM(recorded_asset_value IS NULL) AS missing_asset_value,
        SUM(recorded_total_budget IS NULL) AS missing_total_budget,
        SUM(
            (recorded_asset_value IS NULL)
            != (recorded_total_budget IS NULL)
        ) AS mismatched_missingness
    FROM team_race
    GROUP BY race_number
    ORDER BY race_number
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    valuation_coverage = pd.read_sql_query(query, con)

valuation_coverage

,race_number,team_count,missing_asset_value,missing_total_budget,mismatched_missingness
0,1,15,0,0,0
1,2,15,3,3,0
2,3,15,6,6,0
3,4,15,6,6,0
4,5,15,6,6,0
5,6,15,6,6,0
6,7,15,5,5,0
7,8,15,7,7,0
8,9,15,9,9,0
9,10,15,12,12,0


In [14]:
query = """
    SELECT
        race_number,
        team_name,
        recorded_asset_value,
        remaining_budget,
        recorded_total_budget,
        ROUND(
            recorded_asset_value + remaining_budget
            - recorded_total_budget,
            4
        ) AS difference
    FROM team_race
    WHERE recorded_asset_value IS NOT NULL
      AND recorded_total_budget IS NOT NULL
      AND ABS(
          recorded_asset_value + remaining_budget
          - recorded_total_budget
      ) > 0.01
    ORDER BY race_number, team_name
"""

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    budget_mismatches = pd.read_sql_query(query, con)

budget_mismatches

,race_number,team_name,recorded_asset_value,remaining_budget,recorded_total_budget,difference


## Check Valid Values

In [15]:
queries = {
    "asset types": """
        SELECT asset_type, COUNT(*) AS rows
        FROM asset_race
        GROUP BY asset_type
    """,
    "captain flags": """
        SELECT is_captain, is_mg_captain, COUNT(*) AS rows
        FROM team_race_asset
        GROUP BY is_captain, is_mg_captain
    """,
    "team numeric ranges": """
        SELECT
            MIN(race_rank) AS min_rank,
            MIN(remaining_budget) AS min_cash,
            MIN(transfers_made) AS min_transfers,
            MAX(transfers_made) AS max_transfers,
            MIN(free_transfers_remaining) AS min_free_transfers,
            MAX(free_transfers_remaining) AS max_free_transfers
        FROM team_race
    """,
    "asset price range": """
        SELECT MIN(value) AS min_price, MAX(value) AS max_price
        FROM asset_race
    """,
}

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    for name, query in queries.items():
        print(name)
        display(pd.read_sql_query(query, con))

asset types


,asset_type,rows
0,CONSTRUCTOR,176
1,DRIVER,362


captain flags


,is_captain,is_mg_captain,rows
0,0,0,1345
1,0,1,6
2,1,0,225


team numeric ranges


,min_rank,min_cash,min_transfers,max_transfers,min_free_transfers,max_free_transfers
0,1,0.0,0,5,-3,3


asset price range


,min_price,max_price
0,3.0,33.8


In [16]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    negative_free_transfers = pd.read_sql_query("""
        SELECT
            race_number,
            team_name,
            transfers_made,
            free_transfers_remaining
        FROM team_race
        WHERE free_transfers_remaining < 0
        ORDER BY race_number, team_name
    """, con)

negative_free_transfers

,race_number,team_name,transfers_made,free_transfers_remaining
0,2,RossDonalds motorsport,4,-2
1,4,Gorąco Polecam,4,-1
2,4,RossDonalds motorsport,3,-1
3,4,Trulli madly deeply,5,-2
4,5,"No Michael, no! That's...",3,-1
5,5,real and true max fanclub,3,-1
6,6,Trulli madly deeply,5,-3
7,6,real and true max fanclub,4,-2
8,9,Trulli madly deeply,4,-2
9,12,Ramen Face Racing,4,-1


In [17]:
with sqlite3.connect("../../data/f1_fantasy.db") as con:
    captain_anomalies = pd.read_sql_query("""
        SELECT
            season,
            race_number,
            user_guid,
            team_no,
            SUM(is_captain) AS captain_count
        FROM team_race_asset
        GROUP BY season, race_number, user_guid, team_no
        HAVING SUM(is_captain) != 1
    """, con)

captain_anomalies

,season,race_number,user_guid,team_no,captain_count


In [18]:
queries = {
    "boosted constructors": """
        SELECT t.*
        FROM team_race_asset t
        JOIN asset_race a
          ON a.season = t.season
         AND a.race_number = t.race_number
         AND a.asset_id = t.asset_id
        WHERE a.asset_type != 'DRIVER'
          AND (t.is_captain = 1 OR t.is_mg_captain = 1)
    """,
    "multiple x3 selections": """
        SELECT season, race_number, user_guid, team_no,
               SUM(is_mg_captain) AS x3_count
        FROM team_race_asset
        GROUP BY season, race_number, user_guid, team_no
        HAVING SUM(is_mg_captain) > 1
    """,
}

with sqlite3.connect("../../data/f1_fantasy.db") as con:
    for name, query in queries.items():
        print(name)
        display(pd.read_sql_query(query, con))

boosted constructors


,season,race_number,user_guid,team_no,team_name,asset_id,is_captain,is_mg_captain,player_position,is_final


multiple x3 selections


,season,race_number,user_guid,team_no,x3_count
